# Caso Práctico 3 – Question Answering con Transformers (SQuAD)
Este notebook muestra cómo usar modelos preentrenados de *Hugging Face* (como `bert-large-uncased-whole-word-masking-finetuned-squad`) para responder preguntas a partir de un texto.  
También se incluye una evaluación básica con el conjunto de datos SQuAD.

In [1]:
!pip install protobuf==4.25.3 --force-reinstall

  Using cached protobuf-4.25.3-cp310-abi3-win_amd64.whl.metadata (541 bytes)
Using cached protobuf-4.25.3-cp310-abi3-win_amd64.whl (413 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3


In [2]:
# --- Instalación de dependencias ---
!pip install transformers datasets torch --quiet

In [3]:
!pip install hf_xet

In [4]:
# --- Importación de librerías ---
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator
from datasets import load_dataset
import evaluate
import numpy as np
import torch

c:\Users\jlvar\anaconda3\envs\PLNenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# --- Cargar modelo y tokenizador ---
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
# --- Crear pipeline de Question Answering ---
qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)

Device set to use cpu


In [7]:
# --- Ejemplo de inferencia manual ---
contexto = """
Natural language processing (NLP) is a field of artificial intelligence that gives machines 
the ability to read, understand, and derive meaning from human languages.
"""
pregunta = "What does NLP enable machines to do?"

respuesta = qa_pipeline(question=pregunta, context=contexto)
print(f"Pregunta: {pregunta}")
print(f"Respuesta: {respuesta['answer']}")

Pregunta: What does NLP enable machines to do?
Respuesta: read, understand, and derive meaning from human languages


In [8]:
# --- Cargar un subconjunto del dataset SQuAD ---
dataset = load_dataset("squad", split="validation[:5]")
print(dataset[0])

{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented the AFC at Super Bowl 50?', 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'ans

In [ ]:
for i, ejemplo in enumerate(dataset):
    pregunta = ejemplo["question"]
    contexto = ejemplo["context"]
    respuesta_real = ejemplo["answers"]["text"][0]

    salida = qa_pipeline(question=pregunta, context=contexto)
    print(f"\nEjemplo {i+1}")
    print(f" Pregunta: {pregunta}")
    print(f" Respuesta real: {respuesta_real}")
    print(f" Respuesta modelo: {salida['answer']}")


Ejemplo 1
❓ Pregunta: Which NFL team represented the AFC at Super Bowl 50?
🧾 Respuesta real: Denver Broncos
🤖 Respuesta modelo: Denver Broncos

Ejemplo 2
❓ Pregunta: Which NFL team represented the NFC at Super Bowl 50?
🧾 Respuesta real: Carolina Panthers
🤖 Respuesta modelo: Carolina Panthers

Ejemplo 3
❓ Pregunta: Where did Super Bowl 50 take place?
🧾 Respuesta real: Santa Clara, California
🤖 Respuesta modelo: Levi's Stadium in the San Francisco Bay Area at Santa Clara, California

Ejemplo 4
❓ Pregunta: Which NFL team won Super Bowl 50?
🧾 Respuesta real: Denver Broncos
🤖 Respuesta modelo: Denver Broncos

Ejemplo 5
❓ Pregunta: What color was used to emphasize the 50th anniversary of the Super Bowl?
🧾 Respuesta real: gold
🤖 Respuesta modelo: gold


In [10]:
# Cargar dataset completo (train y validation)
dataset = load_dataset("squad")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [11]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)


Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


###  Preprocesamiento de datos para Question Answering

En esta celda se define la función `preprocess_function`, encargada de preparar el conjunto de datos SQuAD para el entrenamiento del modelo de *Question Answering*.  
El objetivo es transformar las preguntas (`question`) y los textos contextuales (`context`) en representaciones numéricas (tokens) adecuadas para el modelo.

####  Parámetros principales
- **`max_length = 384`** → longitud máxima permitida para cada secuencia de entrada.  
  Los textos más largos se recortan hasta esta longitud.
- **`doc_stride = 128`** → define el grado de solapamiento entre fragmentos cuando un contexto es más largo que `max_length`.  
  Esto permite cubrir posibles respuestas que estén divididas entre dos fragmentos.

####  Proceso de tokenización
La función utiliza el `tokenizer` del modelo preentrenado para convertir texto en tokens y manejar:
- **Truncamiento**: corta los textos largos solo en la segunda secuencia (`only_second`, que corresponde al contexto).  
- **Padding**: añade tokens vacíos para igualar la longitud de todas las secuencias.  
- **Overflowing tokens y stride**: divide contextos extensos en fragmentos solapados para no perder información.

#### Cálculo de posiciones de la respuesta
Cada ejemplo de entrenamiento incluye una o más respuestas correctas dentro del texto.  
El modelo debe aprender a predecir las posiciones de inicio (`start_positions`) y fin (`end_positions`) de la respuesta dentro del contexto tokenizado.

El código:
1. Mapea cada token a su posición original de caracteres mediante `offset_mapping`.
2. Identifica el índice del token `[CLS]` (usado como marcador cuando no hay respuesta).
3. Si la respuesta no se encuentra dentro del fragmento actual, asigna el índice `[CLS]` a ambas posiciones.
4. Si la respuesta está contenida en el fragmento, calcula sus índices de inicio y fin exactos.
5. Asocia esos índices al ejemplo tokenizado para entrenar el modelo a localizar la respuesta correcta.

####  Aplicación al dataset
Finalmente, se aplica la función a todo el conjunto de datos con:
```python
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)


In [12]:
max_length = 384      # longitud máxima por ejemplo
doc_stride = 128      # cantidad de solapamiento entre chunks largos

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = inputs["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = inputs.sequence_ids(i)
        sample_index = sample_mapping[i]
        answer = answers[sample_index]
        if len(answer["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            start_char = answer["answer_start"][0]
            end_char = start_char + len(answer["text"][0])
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                start_positions.append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
tokenized_datasets


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 88524
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 10784
    })
})

In [13]:
# --- Dividir dataset y tomar solo una muestra (5%) para correr rápido en CPU ---
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(int(0.01 * len(tokenized_datasets["train"]))))
eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(int(0.01 * len(tokenized_datasets["validation"]))))

# --- Data collator (mantiene consistencia en padding y batches) ---
data_collator = default_data_collator

print(f"📊 Tamaño del conjunto de entrenamiento: {len(train_dataset)}")
print(f"📊 Tamaño del conjunto de validación: {len(eval_dataset)}")


📊 Tamaño del conjunto de entrenamiento: 885
📊 Tamaño del conjunto de validación: 107


In [14]:
metric = evaluate.load("squad")

def compute_metrics(p):
    return metric.compute(predictions=p.predictions, references=p.label_ids)

In [15]:
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=1,
    logging_steps=50,
)


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # usa una muestra pequeña para probar
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)
trainer.train()


C:\Users\jlvar\AppData\Local\Temp\ipykernel_24348\564522959.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\jlvar\anaconda3\envs\PLNenv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,5.072400
100,4.014500
150,3.948900
200,3.872800


TrainOutput(global_step=222, training_loss=4.194416011775936, metrics={'train_runtime': 1253.1764, 'train_samples_per_second': 0.706, 'train_steps_per_second': 0.177, 'total_flos': 86720995146240.0, 'train_loss': 4.194416011775936, 'epoch': 1.0})

In [ ]:
results = trainer.evaluate()
print("📊 Resultados de evaluación:")
print(results)